In [1]:
import importlib
import spark_uc

importlib.reload(spark_uc)  # pick up spark_uc.py edits without a kernel restart
from spark_uc import create_spark_session, print_session_info

spark = create_spark_session(app_name="SparkTransformations")
print_session_info(spark)  # prints Spark UI URL — open it while this kernel stays running


:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
io.unitycatalog#unitycatalog-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-25ca4249-b3b6-4be4-984e-9397d108c4c3;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found io.unitycatalog#unitycatalog-spark_2.12;0.2.1 in central
	found io.unitycatalog#unitycatalog-client;0.2.1 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.23.1 in central
	found org.apache.logging.log4j#log4j-api;2.23.1 in central
	found org.apache.logging.log4j#log4j-core;2.23.1 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jsr310;2.17.0 in central
	found org.openapitools#jackson-databind-nullable;0.2.6 in central
	found com.googl

Spark session ready.
Unity Catalog URI: http://unity-catalog:8081
Default catalog: unity
Spark application UI: http://127.0.0.1:4040
Spark UI status: reachable (keep this notebook kernel running).
Notebook display() patched for Spark DataFrames.


In [2]:
spark.sparkContext.setLogLevel("ERROR")

df = spark.read.json("/data/source/motor_vehicle_collisions.json")

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("path", "file:///data/tables/collisions_final")
    .saveAsTable("demo.collisions_final")
)

In [3]:
df = spark.sql("select * from demo.collisions_final")

In [4]:
from spark_uc import display

display(df.limit(3))

,PK,_corrupt_record,borough,collision_id,contributing_factor_vehicle_1,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,crash_date,...,number_of_persons_injured,number_of_persons_killed,off_street_name,on_street_name,vehicle_type_code_1,vehicle_type_code_2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5,zip_code
0,1249846,None,None,3498347,Following Too Closely,Unspecified,None,None,None,08/11/2016,...,0,0,None,CROSS ISLAND PARKWAY,Sedan,Station Wagon/Sport Utility Vehicle,None,None,None,NaN
1,1249847,None,MANHATTAN,3502811,Turning Improperly,None,None,None,None,08/14/2016,...,4,0,None,12 AVENUE,Sedan,None,None,None,None,10019.0
2,1249848,None,None,3496571,Unspecified,Unspecified,Unspecified,None,None,08/08/2016,...,1,0,None,34 STREET,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,None,None,NaN


In [7]:
df.count()

2264591

In [6]:
from pyspark.sql.functions import *
df.filter((col("contributing_factor_vehicle_1") == "Following Too Closely") \
          |
         (col("zip_code").isNotNull())
         ).count()

6567672

In [14]:
display(df.limit(10))

,PK,_corrupt_record,borough,collision_id,contributing_factor_vehicle_1,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,crash_date,...,number_of_persons_injured,number_of_persons_killed,off_street_name,on_street_name,vehicle_type_code_1,vehicle_type_code_2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5,zip_code
0,2092298,None,None,130033,Unspecified,Unspecified,None,None,None,07/15/2012,...,0,0,PARKING LOT @ 1080 MC DONALD AVE,None,PASSENGER VEHICLE,UNKNOWN,None,None,None,NaN
1,2088924,None,None,245923,Driver Inattention/Distraction,Unspecified,None,None,None,07/24/2012,...,0,0,W'STONE EXPY - 31 AVE,None,SPORT UTILITY / STATION WAGON,AMBULANCE,None,None,None,NaN
2,2087119,None,None,246003,Unspecified,Unspecified,None,None,None,07/30/2012,...,0,0,R/O BAYTERRACE PKG.LOT 212-45 26 AVE,None,PASSENGER VEHICLE,SMALL COM VEH(4 TIRES),None,None,None,NaN
3,2090982,None,None,246025,Unspecified,Unspecified,None,None,None,07/31/2012,...,0,0,SIDEWALK & FENCE C.I.P. & 150 ST,None,SPORT UTILITY / STATION WAGON,SPORT UTILITY / STATION WAGON,None,None,None,NaN
4,2099337,None,None,246063,Unspecified,None,None,None,None,08/02/2012,...,0,0,F/O 160-26 CROSS ISLAND PKWY SERV.RD,None,PASSENGER VEHICLE,None,None,None,None,NaN
5,2088422,None,None,2888535,Unspecified,Unspecified,None,None,None,08/02/2012,...,0,0,None,None,SPORT UTILITY / STATION WAGON,PASSENGER VEHICLE,None,None,None,NaN
6,2092213,None,None,2896757,Fatigued/Drowsy,Unspecified,None,None,None,07/14/2012,...,0,0,None,None,SPORT UTILITY / STATION WAGON,PASSENGER VEHICLE,None,None,None,NaN
7,2096125,None,None,2896809,Traffic Control Disregarded,Turning Improperly,None,None,None,08/07/2012,...,0,0,None,None,PASSENGER VEHICLE,VAN,None,None,None,NaN
8,2095715,None,None,2926979,Driver Inattention/Distraction,None,None,None,None,07/30/2012,...,0,0,None,None,SPORT UTILITY / STATION WAGON,None,None,None,None,NaN
9,2083925,None,None,3023371,Driver Inattention/Distraction,Unspecified,None,None,None,08/02/2012,...,0,0,None,None,SPORT UTILITY / STATION WAGON,PASSENGER VEHICLE,None,None,None,NaN


In [15]:
df.printSchema()

root
 |-- PK: long (nullable = true)
 |-- _corrupt_record: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- collision_id: long (nullable = true)
 |-- contributing_factor_vehicle_1: string (nullable = true)
 |-- contributing_factor_vehicle_2: string (nullable = true)
 |-- contributing_factor_vehicle_3: string (nullable = true)
 |-- contributing_factor_vehicle_4: string (nullable = true)
 |-- contributing_factor_vehicle_5: string (nullable = true)
 |-- crash_date: string (nullable = true)
 |-- crash_time: string (nullable = true)
 |-- cross_street_name: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- location: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- number_of_cyclist_injured: long (nullable = true)
 |-- number_of_cyclist_killed: long (nullable = true)
 |-- number_of_motorist_injured: long (nullable = true)
 |-- number_of_motorist_killed: long (nullable = true)
 |-- number_of_pedestrians_injured: long (nullable = tru

In [21]:
from pyspark.sql.functions import *
new_df = df.filter((col("contributing_factor_vehicle_1") == "Following Too Closely") \
          |
         (col("zip_code").isNotNull())
         ).select("zip_code","crash_date","collision_id")
display(new_df.limit(5))

,zip_code,crash_date,collision_id
0,10017,09/06/2012,32673
1,10010,09/04/2012,22120
2,10014,08/30/2012,575
3,11229,09/02/2012,116740
4,11104,08/16/2012,239125


In [30]:
final_df = new_df.filter(
    to_date(col("crash_date"), "MM/dd/yyyy") == to_date(lit("09/06/2012"), "MM/dd/yyyy")
)
display(final_df.limit(10))


,zip_code,crash_date,collision_id
0,10017,09/06/2012,32673
1,10010,09/06/2012,22137
2,11238,09/06/2012,191826
3,10025,09/06/2012,57789
4,11233,09/06/2012,160483
5,10025,09/06/2012,57790
6,11233,09/06/2012,182883
7,10005,09/06/2012,632
8,11234,09/06/2012,126142
9,11355,09/06/2012,246621


In [31]:
display(
    new_df.select(
        "crash_date",
        to_date(col("crash_date"), "yyyy-MM-dd").alias("wrong_parse"),
        to_date(col("crash_date"), "MM/dd/yyyy").alias("correct_parse"),
    ).limit(5)
)


,crash_date,wrong_parse,correct_parse
0,09/06/2012,None,2012-09-06
1,09/04/2012,None,2012-09-04
2,08/30/2012,None,2012-08-30
3,09/02/2012,None,2012-09-02
4,08/16/2012,None,2012-08-16


26/06/07 09:05:32 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/06/07 11:10:51 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.6: worker lost: Not receiving heartbeat for 60 seconds
